# DepthWizard — Part 2: Depth Inference (Colab)Runs Depth Anything V2 on your ingested scenes and gives you back `depth.npy`.**Runtime → Change runtime type → T4 GPU** before you start. On CPU this is ~40x slower.Everything after this notebook runs locally in VS Code. This is the only GPU step in the whole prototype.

In [ ]:
import torchprint("CUDA:", torch.cuda.is_available())print("GPU :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - set Runtime > Change runtime type > T4 GPU")

## 1. Install Depth Anything V2

In [ ]:
!git clone -q https://github.com/DepthAnything/Depth-Anything-V2%cd Depth-Anything-V2!pip install -q -r requirements.txtprint("installed")

## 2. Download the checkpoint`vitl` (335 MB) is the best quality and fits comfortably on a T4. Drop to `vitb` only if you hit memory limits.

In [ ]:
ENCODER = "vitl"   # vits | vitb | vitl!mkdir -p checkpoints!wget -q --show-progress -O checkpoints/depth_anything_v2_{ENCODER}.pth \  https://huggingface.co/depth-anything/Depth-Anything-V2-Large/resolve/main/depth_anything_v2_vitl.pthimport osprint("checkpoint MB:", round(os.path.getsize(f"checkpoints/depth_anything_v2_{ENCODER}.pth")/1e6, 1))

## 3. Upload your scenesUpload the `rgb.npy` files your ingest stage produced. They live in`data/work/<scene>/rgb.npy` on your machine.**Rename each one before uploading** so you can tell them apart — e.g.`synthetic__rgb.npy`, `morocco__rgb.npy`. The double underscore is thescene-name separator this notebook expects.Upload `synthetic__rgb.npy` at minimum — it is your correctness check.

In [ ]:
import osos.makedirs("/content/scenes", exist_ok=True)%cd /content/scenesfrom google.colab import filesuploaded = files.upload()%cd /content/Depth-Anything-V2import globscene_files = sorted(glob.glob("/content/scenes/*__rgb.npy"))print("\nfound:")for f in scene_files:    print("  ", os.path.basename(f))if not scene_files:    print("  NONE - filenames must end in __rgb.npy")

## 4. Load the model

In [ ]:
import torch, numpy as np, cv2, timefrom depth_anything_v2.dpt import DepthAnythingV2MODEL_CONFIGS = {    "vits": {"encoder": "vits", "features": 64,  "out_channels": [48, 96, 192, 384]},    "vitb": {"encoder": "vitb", "features": 128, "out_channels": [96, 192, 384, 768]},    "vitl": {"encoder": "vitl", "features": 256, "out_channels": [256, 512, 1024, 1024]},}DEVICE = "cuda" if torch.cuda.is_available() else "cpu"model = DepthAnythingV2(**MODEL_CONFIGS[ENCODER])model.load_state_dict(torch.load(f"checkpoints/depth_anything_v2_{ENCODER}.pth", map_location="cpu"))model = model.to(DEVICE).eval()print("model ready on", DEVICE)

## 5. Run inference`infer_image` expects **BGR** (OpenCV convention). Our `rgb.npy` is RGB, so we flipthe channel order. Getting this wrong doesn't crash — it just quietly degrades theoutput, which is worse.`input_size=518` is the model's native resolution. Raising it to 1036 sharpensbuilding edges noticeably at ~4x the time. Both are cheap here; try 518 first.

In [ ]:
INPUT_SIZE = 518def run_scene(path):    scene = os.path.basename(path).split("__")[0]    rgb = np.load(path)    if rgb.dtype != np.uint8:        rgb = np.clip(rgb, 0, 255).astype(np.uint8)    bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)    t0 = time.time()    depth = model.infer_image(bgr, input_size=INPUT_SIZE)    dt = time.time() - t0    depth = np.asarray(depth, dtype=np.float32)    out = f"/content/scenes/{scene}__depth.npy"    np.save(out, depth)    print(f"{scene:<24} {rgb.shape[1]}x{rgb.shape[0]}  ->  depth "          f"{depth.shape[1]}x{depth.shape[0]}  "          f"range [{depth.min():.3f}, {depth.max():.3f}]  {dt:.1f}s")    return scene, rgb, depthresults = [run_scene(p) for p in scene_files]

## 6. Eyeball itLook for: buildings brighter than ground (the model puts *nearer* = higher value,and from nadir, nearer means taller). Roofs should be flat-ish plateaus, not domes.If buildings come out **darker** than the ground, the sign is flipped — don't fix ithere. The local `depth_check` stage detects and records that automatically.

In [ ]:
import matplotlib.pyplot as pltfor scene, rgb, depth in results:    fig, ax = plt.subplots(1, 2, figsize=(13, 6))    ax[0].imshow(rgb);                 ax[0].set_title(f"{scene} RGB");   ax[0].axis("off")    im = ax[1].imshow(depth, cmap="turbo")    ax[1].set_title(f"{scene} raw depth");  ax[1].axis("off")    fig.colorbar(im, ax=ax[1], fraction=0.046)    plt.tight_layout(); plt.show()

## 7. DownloadUnzip into `data/work/` locally so each `<scene>__depth.npy` lands next to the`rgb.npy` it came from. The next script expects `data/work/<scene>/depth.npy`.

In [ ]:
!cd /content/scenes && zip -q -r /content/depth_outputs.zip *__depth.npyfrom google.colab import filesfiles.download("/content/depth_outputs.zip")